# Pemeriksaan perbaikan field SLR_09 v0.8.2

Replay deterministic sudah dijalankan aktual melalui replay_field_repair_v082.py sebelum notebook ini. Notebook memeriksa bukti tersebut dan hasil76 tes kontrak yang sudah dijalankan, lalu menjalankan tes evaluator. Tidak ada neural inference, gradient, atau acuan baru yang diinferensikan di sini.50 acuan FINAL081 yang dipakai memperbaiki kode kini development/regresi.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
WORK=Path(os.environ['SLR09_WORK_DIR'])
state=json.loads((ROOT/'evidence/field_fix_v082/replay_fixed/run_state.json').read_text())
contract=json.loads((ROOT/'evidence/field_fix_v082/contract_tests/active_tests.json').read_text())
assert state['status']=='SUCCEEDED' and contract['all_passed'] and contract['tests_run']==76
for folder in ['reviewed50','previous50']:
 p=ROOT/'evidence/field_fix_v082/replay_fixed'/folder
 e=json.loads((p/'evaluation.json').read_text())
 assert e['replay_predictions_sha256']==hashlib.sha256((p/'predictions.jsonl').read_bytes()).hexdigest()
 assert e['metric_targets_met'] and not e['quality_acceptance_approved']
print(json.dumps({'replay_actual_cli_evidence':state,'contract_actual_cli_evidence':contract},ensure_ascii=False,indent=2))


In [ ]:
result=subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'code/acceptance'),'-p','test_*.py','-v'],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
(ROOT/'evidence/field_fix_v082/evaluator_tests.txt').write_text(result.stdout)
if result.returncode:raise RuntimeError('Evaluator tests failed; original output retained.')
print('Semua hasil replay adalah regresi; acuan FINAL082 baru tetap PENDING.')
